<a href="https://colab.research.google.com/github/aaagairi/Projexa-Pairadox/blob/main/surv_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# imports

!pip install torch torchaudio transformers librosa audiomentations tqdm scikit-learn

import os
import torch
import torchaudio
import librosa
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import ASTFeatureExtractor, ASTForAudioClassification

from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.0.0
    Uninstalling soxr-1.0.0:
      Successfully uninstalled soxr-1.0.0


In [ ]:
# audio augmentation

augment = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.5),
    TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
    PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
    Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
])

In [ ]:
# smart audio loading

def load_audio_segment(file_path, sr=16000, duration=5):

    audio, sample_rate = librosa.load(file_path, sr=sr)

    segment_length = sr * duration

    if len(audio) <= segment_length:
        padding = segment_length - len(audio)
        audio = np.pad(audio, (0, padding))
        return audio

    max_start = len(audio) - segment_length
    start = np.random.randint(0, max_start)

    return audio[start:start+segment_length]

In [ ]:
# Dataset

class AudioDataset(Dataset):

    def __init__(self, files, labels, feature_extractor, augment=False):
        self.files = files
        self.labels = labels
        self.feature_extractor = feature_extractor
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        audio = load_audio_segment(self.files[idx])

        if self.augment:
            audio = augment(samples=audio, sample_rate=16000)

        inputs = self.feature_extractor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs["input_values"].squeeze()

        label = torch.tensor(self.labels[idx])

        return input_values, label

In [ ]:
data_dir = "/content/drive/MyDrive/Surveillance Project/Surveillance Project/Audio"


# 1 Build dataset lists
files = []
labels = []

label_map = {
    "Ambient":0,
    "Contextual":1,
    "Suspicious":2
}

for main_class in label_map:
    main_folder = os.path.join(data_dir, main_class)

    for root, dirs, file_list in os.walk(main_folder):
        for file in file_list:
            if file.lower().endswith(".wav"):
                path = os.path.join(root, file)

                files.append(path)
                labels.append(label_map[main_class])

print("Total files:", len(files))

Total files: 179


In [ ]:
# split

from sklearn.model_selection import train_test_split

train_files, val_files, train_labels, val_labels = train_test_split(
    files,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [ ]:
# load audio transformer (AST)

feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=3,
    ignore_mismatched_sizes=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                       
------------------------+----------+---------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([3, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
# dataloaders

train_dataset = AudioDataset(train_files, train_labels, feature_extractor, augment=True)
val_dataset = AudioDataset(val_files, val_labels, feature_extractor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
# optimizer

optimizer = optim.AdamW(model.parameters(), lr=3e-5)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [ ]:
# training loop

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

criterion = nn.CrossEntropyLoss()

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for inputs, labels in tqdm(train_loader):

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    print("Epoch:", epoch, "Loss:", total_loss/len(train_loader))

100%|██████████| 18/18 [02:13<00:00,  7.42s/it]


Epoch: 0 Loss: 0.3540868092742231


100%|██████████| 18/18 [00:47<00:00,  2.64s/it]


Epoch: 1 Loss: 0.09665695870191687


100%|██████████| 18/18 [00:49<00:00,  2.77s/it]


...

Epoch: 7 Loss: 0.00531234444416542


100%|██████████| 18/18 [00:50<00:00,  2.80s/it]


Epoch: 8 Loss: 0.009353009156054921


100%|██████████| 18/18 [00:49<00:00,  2.77s/it]

Epoch: 9 Loss: 0.006970497061653684


In [ ]:
# eval

model.eval()

preds = []
true = []

with torch.no_grad():

    for inputs, labels in val_loader:

        inputs = inputs.to(device)

        outputs = model(inputs).logits

        p = torch.argmax(outputs, dim=1).cpu().numpy()

        preds.extend(p)
        true.extend(labels.numpy())

print(classification_report(true, preds))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97        18
           1       0.94      1.00      0.97        17
           2       1.00      1.00      1.00         1

    accuracy                           0.97        36
   macro avg       0.98      0.98      0.98        36
weighted avg       0.97      0.97      0.97        36



In [ ]:
# inference function

def predict_audio(file_path):

    audio = load_audio_segment(file_path)

    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():

        logits = model(inputs["input_values"].to(device)).logits

    pred = torch.argmax(logits, dim=1).item()

    labels = ["Ambient","Contextual","Suspicious"]

    return labels[pred]